<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab11: 데이터마이닝 — 의사결정나무, 연관분석, 텍스트마이닝

본 실습 자료는 『배워서 바로 써먹는 데이터 분석 with 파이썬』 (설진욱, 생능북스)의 내용을 참고하여 제작되었습니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from math import log2

from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

### 한글 폰트 설정

In [ ]:
!apt-get install -y fonts-nanum

In [ ]:
import platform

if platform.system() == 'Darwin':
    mpl.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    mpl.rc('font', family='Malgun Gothic')
else:
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    mpl.rc('font', family='NanumGothic')

mpl.rc('axes', unicode_minus=False)
print('폰트 설정 완료:', mpl.rcParams['font.family'])

---
## PART 1. 의사결정나무 (Decision Tree)

**핵심 아이디어**: 데이터를 분기 조건으로 재귀적으로 분할하여 **분류 또는 예측**을 수행하는 트리 모델

- **루트 노드**: 최상위 분기 조건
- **내부 노드**: 중간 분기 조건
- **리프 노드**: 최종 예측값 (클래스 또는 확률)

각 분기는 **불순도(Impurity)가 가장 많이 감소**하는 변수와 기준값으로 결정한다.

- **지니 계수**
  - $G = 1 - \sum_{k=1}^{C} p_k^2 = 2p(1-p)$
- **엔트로피**
  - $H = -\sum_{k=1}^{C} p_k \log_2 p_k$


### 1-1. 데이터 준비 — Iris 붓꽃 데이터셋

- 표본: 150개, 붓꽃 **3가지 품종** (Setosa / Versicolor / Virginica)
- 변수: **4개** (꽃받침 길이·너비, 꽃잎 길이·너비)
- 목표: 측정값으로 품종 분류

In [ ]:
iris = load_iris()

feature_names_kr = ['꽃받침 길이(cm)', '꽃받침 너비(cm)', '꽃잎 길이(cm)', '꽃잎 너비(cm)']
species_names    = ['세토사', '버시컬러', '버지니카']

df = pd.DataFrame(iris.data, columns=feature_names_kr)
df['품종코드'] = iris.target
df['품종명']   = df['품종코드'].map({0: '세토사', 1: '버시컬러', 2: '버지니카'})
df.head()

In [ ]:
df.shape

In [ ]:
# 품종별 샘플 수
df['품종명'].value_counts().sort_index()

### 1-2. 불순도(Impurity) 시각화 — 지니 계수 vs 엔트로피

이진 분류 ($C = 2$)에서 클래스 1의 비율 $p$에 따른 불순도 변화.
- $p = 0.5$: 두 클래스가 반반 → **최대 불순도**
- $p = 0$ 또는 $p = 1$: 한 클래스만 존재 → **순수 노드 (불순도 = 0)**

In [ ]:
p[1]

In [ ]:
p = np.linspace(0.001, 0.999, 400)
gini           = 2 * p * (1 - p)
entropy        = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
misclassify    = 1 - np.maximum(p, 1 - p)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p, gini,        color='crimson',   linewidth=2,   linestyle='-',  label=r'지니 계수 $G = 2p(1-p)$')
ax.plot(p, entropy,     color='black',     linewidth=2,   linestyle='--', label=r'엔트로피 $H = -p\log_2 p - (1-p)\log_2(1-p)$')
ax.plot(p, misclassify, color='steelblue', linewidth=1.5, linestyle=':',  label=r'오분류율 $1 - \max(p, 1-p)$')

ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(0.52, 0.9, 'p = 0.5\n(최대 불순도)', fontsize=9, color='gray')

ax.set_xlabel('클래스 1의 비율 p', fontsize=12)
ax.set_ylabel('불순도', fontsize=12)
ax.set_title('불순도 지표 비교 — 지니 계수 vs 엔트로피', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("엔트로피가 지니보다 더 민감하게 반응하지만, 실무 차이는 크지 않음")
print("→ sklearn 기본값: criterion='gini'")

### 1-3. 의사결정나무 모델 학습

데이터를 훈련(80%)·테스트(20%)로 분리 후 `DecisionTreeClassifier`로 학습한다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
X = df[feature_names_kr].values
y = df['품종코드'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(criterion='gini', random_state=42)
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
test_acc  = clf.score(X_test,  y_test)
print(f"훈련 정확도: {train_acc:.4f}")
print(f"테스트 정확도: {test_acc:.4f}")
print(f"트리 깊이: {clf.get_depth()}")
print(f"리프 노드 수: {clf.get_n_leaves()}")

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=species_names))

### 1-4. 트리 시각화

`plot_tree()`로 학습된 트리 구조를 직접 시각화한다.  
각 노드에는 분기 조건, 지니 계수, 샘플 수, 클래스 분포가 표시된다.

In [ ]:
from sklearn.tree import plot_tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    clf,
    feature_names=feature_names_kr,
    class_names=species_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('의사결정나무 — Iris 분류 (가지치기 없음)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import plot_tree, export_text
print(export_text(clf, feature_names=feature_names_kr))

### 1-5. 과적합과 가지치기 — max_depth 비교

트리가 너무 깊어지면 훈련 데이터의 노이즈까지 학습 → **과적합**  
- 훈련 정확도 ↑, 테스트 정확도 ↓ → **일반화 실패**
- `max_depth` 제한으로 사전 가지치기(Pre-Pruning) 적용

In [ ]:
depths     = list(range(1, 9)) + [None]
train_accs = []
test_accs  = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(m.score(X_train, y_train))
    test_accs.append(m.score(X_test,  y_test))

x_labels = [str(d) if d is not None else 'None\n(무제한)' for d in depths]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(depths)), train_accs, 'o-', color='crimson',  linewidth=2, label='훈련 정확도')
ax.plot(range(len(depths)), test_accs,  's-', color='steelblue', linewidth=2, label='테스트 정확도')
ax.set_xticks(range(len(depths)))
ax.set_xticklabels(x_labels, fontsize=9)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('정확도', fontsize=12)
ax.set_title('max_depth에 따른 훈련/테스트 정확도 비교', fontsize=13)
ax.set_ylim(0.7, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
clf_pruned = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
clf_pruned.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))

for ax, model, title in zip(
    axes,
    [clf_pruned, clf],
    ['가지치기 적용 (max_depth=3, min_samples_leaf=5)',
     '가지치기 없음 (무제한)']
):
    plot_tree(model, feature_names=feature_names_kr, class_names=species_names,
              filled=True, rounded=True, fontsize=9, ax=ax)
    acc = model.score(X_test, y_test)
    ax.set_title(f"{title}\n테스트 정확도: {acc:.4f}", fontsize=11)

plt.suptitle('가지치기 전후 트리 구조 비교', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 1-6. 특성 중요도 (Feature Importance)

각 변수가 전체 불순도 감소에 기여한 비율.  
- 값이 클수록 분류에 더 중요한 변수
- 값의 합 = 1.0

In [ ]:
fi = pd.Series(clf_pruned.feature_importances_, index=feature_names_kr).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(fi.index, fi.values, color='steelblue', edgecolor='navy')

for bar, val in zip(bars, fi.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('특성 중요도', fontsize=12)
ax.set_title('의사결정나무 특성 중요도 (max_depth=3)', fontsize=13)
ax.set_xlim(0, fi.max() * 1.2)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("꽃잎 길이·너비가 품종 분류에 압도적으로 중요함")
print("→ 꽃받침 변수는 중요도 ≈ 0 (분류에 거의 기여하지 않음)")

---
## PART 2. 연관분석 (Association Analysis)

데이터에서 **함께 자주 발생하는 항목들의 관계**를 규칙으로 표현하는 기법.

$$X \Rightarrow Y$$

전건(Antecedent) $X$가 발생하면 후건(Consequent) $Y$도 발생하는 경향

- 지지도 (Support): $\frac{\|X \cap Y\|}{N}$
  - $X$와 $Y$ 동시 발생 빈도
- 신뢰도 (Confidence): $\frac{\|X \cap Y\|}{\|X\|}$
  - $X$ 구매 시 $Y$ 구매 확률
- 향상도 (Lift): $\frac{\text{Confidence}(X \Rightarrow Y)}{P(Y)}$
  - 독립 대비 연관 강도

**향상도 해석**: Lift > 1 (양의 연관, 유의미한 규칙) / = 1 (독립) / < 1 (음의 연관)

### 2-1. 거래 데이터 구성

강의 슬라이드의 예시 데이터 (T1–T5) 그대로 사용.

In [ ]:
transactions = [
    ['우유', '기저귀', '맥주'],          # T1
    ['우유', '기저귀'],                  # T2
    ['기저귀', '맥주', '콜라'],          # T3
    ['우유', '맥주', '콜라'],            # T4
    ['우유', '기저귀', '맥주', '콜라'],  # T5
]
N     = len(transactions)
items = sorted({'우유', '기저귀', '맥주', '콜라'})

print(f"총 거래 수 (N): {N}")
print(f"전체 아이템: {items}")

In [ ]:
rows = []
for i, t in enumerate(transactions, 1):
    row = {item: ('O' if item in t else '') for item in items}
    row['거래 ID'] = f'T{i}'
    rows.append(row)

tx_df = pd.DataFrame(rows).set_index('거래 ID')[items]
tx_df

### 2-2. 지지도·신뢰도·향상도 직접 계산

**Step 1**: 각 아이템셋의 등장 횟수와 지지도를 계산한다.

In [ ]:
def get_support(itemset, transactions):
    """itemset(frozenset)의 지지도 반환"""
    count = sum(1 for t in transactions if itemset.issubset(set(t)))
    return count / len(transactions)

# 전체 아이템셋 지지도 (1-아이템셋 + 2-아이템셋)
support = {}

for item in items:
    fs = frozenset([item])
    support[fs] = get_support(fs, transactions)

for a, b in combinations(items, 2):
    fs = frozenset([a, b])
    support[fs] = get_support(fs, transactions)

print("아이템셋별 지지도:")
print(f"{'아이템셋':<25} {'지지도':>8}")
print("-" * 36)
for fs, sup in sorted(support.items(), key=lambda x: (-len(x[0]), -x[1])):
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f"{label:<25} {sup:>8.2f}")

**Step 2**: 최소 지지도(0.4) 이상인 아이템셋에서 연관규칙을 생성하고, 신뢰도·향상도를 계산한다.

In [ ]:
min_support    = 0.4
min_confidence = 0.6

rules = []
for itemset, sup_xy in support.items():
    if len(itemset) < 2 or sup_xy < min_support:
        continue
    for size in range(1, len(itemset)):
        for ant_tuple in combinations(sorted(itemset), size):
            antecedent = frozenset(ant_tuple)
            consequent = itemset - antecedent

            sup_x  = support.get(antecedent,  get_support(antecedent,  transactions))
            sup_y  = support.get(consequent,  get_support(consequent,  transactions))
            conf   = sup_xy / sup_x
            lift   = conf   / sup_y

            if conf >= min_confidence:
                rules.append({
                    '전건(X)': ', '.join(sorted(antecedent)),
                    '후건(Y)': ', '.join(sorted(consequent)),
                    '지지도':   round(sup_xy, 2),
                    '신뢰도':   round(conf,   2),
                    '향상도':   round(lift,   2)
                })

rules_df = pd.DataFrame(rules).sort_values('향상도', ascending=False).reset_index(drop=True)
print(f"총 {len(rules_df)}개 규칙 발견 (지지도 ≥ {min_support}, 신뢰도 ≥ {min_confidence})")
rules_df

### 2-3. 예시 검증 — {기저귀} ⇒ {맥주}

In [ ]:
ant = frozenset(['기저귀'])
con = frozenset(['맥주'])
xy  = ant | con

sup_xy = get_support(xy,  transactions)
sup_x  = get_support(ant, transactions)
sup_y  = get_support(con, transactions)
conf   = sup_xy / sup_x
lift   = conf   / sup_y

print("{기저귀} ⇒ {맥주} 규칙 검증")
print(f"  기저귀 포함 거래: T1, T2, T3, T5  → |X| = {int(sup_x * N)}")
print(f"  기저귀 & 맥주 동시 포함: T1, T3, T5 → |X∩Y| = {int(sup_xy * N)}")
print(f"  맥주 포함 거래: T1, T3, T4, T5   → P(Y) = {sup_y:.2f}")
print()
print(f"  지지도  = {int(sup_xy * N)}/{N} = {sup_xy:.2f}")
print(f"  신뢰도  = {int(sup_xy * N)}/{int(sup_x * N)} = {conf:.2f}")
print(f"  향상도  = {conf:.2f} / {sup_y:.2f} = {lift:.2f}")
print()
print(f"  → Lift = {lift:.2f} < 1: 독립에 가까운 약한 연관")

### 2-4. 연관규칙 시각화 — 지지도·신뢰도·향상도 산점도

x축: 지지도, y축: 신뢰도, 점 크기·색상: 향상도  
오른쪽 위 + 밝은 색일수록 **좋은 규칙**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    rules_df['지지도'], rules_df['신뢰도'],
    c=rules_df['향상도'], cmap='RdYlGn',
    s=rules_df['향상도'] * 200,
    edgecolors='gray', linewidths=0.5, alpha=0.85
)
plt.colorbar(sc, ax=ax, label='향상도 (Lift)')

for _, row in rules_df.iterrows():
    label = f"{{{row['전건(X)']}}}\n→{{{row['후건(Y)']}}}"
    ax.annotate(label,
                (row['지지도'], row['신뢰도']),
                textcoords='offset points', xytext=(6, 4), fontsize=7)

ax.axhline(min_confidence, color='steelblue', linestyle='--', linewidth=1.2,
           label=f'최소 신뢰도 = {min_confidence}')
ax.axvline(min_support,    color='crimson',   linestyle='--', linewidth=1.2,
           label=f'최소 지지도 = {min_support}')

ax.set_xlabel('지지도 (Support)', fontsize=12)
ax.set_ylabel('신뢰도 (Confidence)', fontsize=12)
ax.set_title('연관규칙 지지도–신뢰도–향상도 산점도', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## PART 3. 텍스트마이닝 (Text Mining)

비정형(unstructured) 텍스트 데이터에서 유용한 정보와 패턴을 추출하는 분석 기법.

**텍스트 처리 파이프라인**: 원문 텍스트 → 토큰화 → 불용어 제거 → 형태소 분석 → **벡터화 (TF-IDF)**

**TF-IDF 핵심 아이디어**:
- 특정 문서에 **자주 등장** → TF(Term Frequency) ↑
- 여러 문서에 **드물게 등장** → IDF(Inverse Document Frequency) ↑
- 두 조건 모두 만족 → 해당 문서의 **핵심 단어**

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

$$\text{TF}(t, d) = \frac{n_{t,d}}{\sum_k n_{k,d}}, \quad \text{IDF}(t) = \log\!\left(\frac{N}{1 + df(t)}\right)$$

- $n_{t,d}$: 문서 $d$에서 단어 $t$의 출현 횟수
- $N$: 전체 문서 수, $df(t)$: $t$가 등장한 문서 수

### 3-1. 코퍼스(Corpus) 구성 — 사전 토큰화된 문서

한국어는 형태소 분석(KoNLPy 등)이 필요하지만, 여기서는 단어가 이미 공백으로 분리된
사전 토큰화(pre-tokenized) 문서를 사용하여 TF-IDF 원리에 집중한다.

In [ ]:
corpus = [
    '데이터 분석 파이썬 머신러닝 데이터',     # 기사1
    '파이썬 프로그래밍 웹 개발 파이썬',       # 기사2
    '머신러닝 딥러닝 인공지능 데이터 학습',   # 기사3
    '데이터 시각화 분석 통계 그래프',         # 기사4
    '인공지능 로봇 자동화 딥러닝 머신러닝',   # 기사5
]
doc_names = ['기사1', '기사2', '기사3', '기사4', '기사5']

print("코퍼스 구성:")
for name, doc in zip(doc_names, corpus):
    print(f"  {name}: {doc}")

### 3-2. TF-IDF 직접 계산 (수식 구현)

수식을 파이썬으로 직접 구현하여 TF-IDF 값이 어떻게 결정되는지 확인한다.

In [ ]:
tokenized = [doc.split() for doc in corpus]
vocab     = sorted(set(w for doc in tokenized for w in doc))

# TF: 문서 내 단어 상대 빈도
tf_matrix = np.zeros((len(corpus), len(vocab)))
for i, doc in enumerate(tokenized):
    for word in doc:
        j = vocab.index(word)
        tf_matrix[i, j] += 1
    tf_matrix[i] /= len(doc)

# IDF: log(N / (1 + df(t)))
df_count = (tf_matrix > 0).sum(axis=0)  # 각 단어가 등장한 문서 수
idf      = np.log(len(corpus) / (1 + df_count))

# TF-IDF
tfidf_matrix = tf_matrix * idf

tfidf_df = pd.DataFrame(tfidf_matrix.round(3), index=doc_names, columns=vocab)
print("TF-IDF 행렬 (행=문서, 열=단어):")
tfidf_df

In [ ]:
print("단어별 IDF 값 (낮을수록 여러 문서에 흔히 등장):")
idf_series = pd.Series(idf.round(3), index=vocab).sort_values(ascending=False)
print(idf_series)
print()
print("→ '데이터'는 3개 기사에 등장 → IDF 낮음 → TF-IDF도 낮음")
print("→ '파이썬'은 2개 기사에만 등장 → 기사2에서 TF-IDF 높음")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    tfidf_df,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': 'TF-IDF'},
    ax=ax
)
ax.set_title('TF-IDF 히트맵 — 문서별 단어 중요도', fontsize=13)
ax.set_xlabel('단어', fontsize=11)
ax.set_ylabel('문서', fontsize=11)
plt.tight_layout()
plt.show()

### 3-3. sklearn TfidfVectorizer 활용

`TfidfVectorizer`는 토큰화·TF-IDF 계산을 한 번에 처리하는 실용 도구.  
sklearn은 평활화된 IDF($\log\frac{1+N}{1+df(t)} + 1$)를 사용하므로 수동 계산과 값이 조금 다르다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(corpus)

feature_words = vectorizer.get_feature_names_out()
tfidf_sk_df   = pd.DataFrame(
    X_tfidf.toarray().round(3),
    index=doc_names,
    columns=feature_words
)
print(f"벡터 차원: {X_tfidf.shape}  (문서 수 × 어휘 수)")
print()
print("TF-IDF 행렬 (sklearn, L2 정규화 적용):")
tfidf_sk_df

In [ ]:
print("각 기사별 TF-IDF 상위 3 단어:")
print("-" * 40)
for doc in doc_names:
    top3 = tfidf_sk_df.loc[doc].nlargest(3)
    words = ', '.join([f'{w}({v:.2f})' for w, v in top3.items()])
    print(f"  {doc}: {words}")

### 3-4. 워드클라우드 (Word Cloud)

단어 빈도(또는 TF-IDF)에 비례하는 크기로 시각화하여 **핵심 키워드를 한눈에 파악**한다.  
한국어 워드클라우드는 반드시 `font_path`에 한글 폰트를 지정해야 한다.

In [ ]:
# TF-IDF 전체 합계 → 단어 중요도
word_importance = tfidf_sk_df.sum(axis=0).sort_values(ascending=False).to_dict()

try:
    from wordcloud import WordCloud

    if platform.system() == 'Darwin':
        wc_font = '/Library/Fonts/NanumGothic.otf'
    elif platform.system() == 'Windows':
        wc_font = 'C:/Windows/Fonts/malgunbd.ttf'
    else:
        wc_font = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

    wc = WordCloud(
        font_path=wc_font,
        width=900, height=450,
        background_color='white',
        colormap='tab10',
        max_words=30
    ).generate_from_frequencies(word_importance)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('TF-IDF 기반 워드클라우드\n(크기 ∝ 전체 문서 TF-IDF 합계)', fontsize=13)
    plt.tight_layout()
    plt.show()

except (ImportError, OSError) as e:
    print(f"워드클라우드 생성 불가: {e}")
    print("설치: pip install wordcloud\n")

    # 대체: 수평 막대 차트
    wi_series = pd.Series(word_importance).sort_values(ascending=True).tail(12)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(wi_series.index, wi_series.values, color='steelblue', edgecolor='navy')
    ax.set_xlabel('TF-IDF 합계')
    ax.set_title('단어별 TF-IDF 합계 (상위 12)')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

---
## 요약

### 핵심 개념

| 주제 | 개념 | 주요 함수/속성 |
|---|---|---|
| **의사결정나무** | 불순도 감소 기준으로 재귀 분기 | `DecisionTreeClassifier` |
| 지니 계수 | $G = 2p(1-p)$, CART 기본값 | `criterion='gini'` |
| 엔트로피 | $H = -\sum p_k \log_2 p_k$, 정보 이득 | `criterion='entropy'` |
| 가지치기 | 과적합 방지 — max_depth, min_samples_leaf | `max_depth=k` |
| 특성 중요도 | 불순도 감소 기여 비율, 합 = 1 | `feature_importances_` |
| **연관분석** | 함께 자주 등장하는 항목 간 규칙 발견 | — |
| 지지도 | $\|X \cup Y\| / N$ — 동시 발생 빈도 | — |
| 신뢰도 | $\|X \cup Y\| / \|X\|$ — 조건부 확률 | — |
| 향상도 | Confidence / P(Y) — 독립 대비 연관 강도 | — |
| Apriori | 단조성 원리 → 비빈발 후보 가지치기 | — |
| **텍스트마이닝** | 비정형 텍스트 → 유용한 정보·패턴 추출 | — |
| TF | 문서 내 단어 상대 빈도 | `TfidfVectorizer` |
| IDF | $\log(N / (1 + df(t)))$ — 희귀 단어 가중치 ↑ | `TfidfVectorizer` |
| 워드클라우드 | TF-IDF 비례 크기로 핵심 키워드 시각화 | `WordCloud` |

### 실습 결과 요약

- **의사결정나무**: Iris 150개 샘플 — `max_depth=3`으로 가지치기 후 테스트 정확도 ≥ 96%, **꽃잎** 변수가 분류에 압도적으로 중요
- **연관분석**: T1–T5 거래 데이터 — 지지도 0.4·신뢰도 0.6 기준 규칙 발견, `{기저귀}→{맥주}` 향상도 < 1 (독립에 가까움)
- **텍스트마이닝**: 5개 기사 코퍼스 — '데이터'처럼 여러 문서에 흔히 등장하는 단어는 TF-IDF가 낮고, '파이썬'처럼 특정 문서에만 집중된 단어는 TF-IDF가 높음

> **한국어 텍스트마이닝 주의**: 실제 한국어 분석에서는 형태소 분석기(KoNLPy: Okt, Mecab 등)로 어근 추출 후 TF-IDF를 적용해야 정확한 결과를 얻을 수 있음  
> **Apriori 한계**: 아이템 수 증가 시 후보 수가 지수적으로 증가 → 대규모 데이터에는 **FP-Growth** 알고리즘 사용 권장